# نوت‌بوک ۲ — تحلیل خطای طبقه‌بند پایه

**هدف:** پیش‌بینی‌های طبقه‌بند پایه را به سه بُعد می‌شکنیم — منبع (انسان / هر سه مولد)، حوزه موضوعی، و طول متن — تا الگوی خطاها را ببینیم.

**ورودی:** `results/baseline_predictions.jsonl` که در نوت‌بوک ۱ ذخیره شد.

**خروجی:** `results/error_analysis.json`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
from collections import defaultdict
import numpy as np

DRIVE_ROOT = Path('/content/drive/MyDrive/persian-ai-research')
RESULTS_DIR = DRIVE_ROOT / 'results'
TEST_PATH = DRIVE_ROOT / 'data' / 'splits' / 'test.jsonl'
PREDS_PATH = RESULTS_DIR / 'baseline_predictions.jsonl'

def load_jsonl(p):
    with open(p, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

test = load_jsonl(TEST_PATH)
preds = load_jsonl(PREDS_PATH)
test_by_id = {r['id']: r for r in test}
print(f'نمونه‌های آزمون: {len(test)} | پیش‌بینی‌ها: {len(preds)}')

In [ ]:
# الحاق هر پیش‌بینی به فراداده نمونه
joined = []
for p in preds:
    r = test_by_id[p['id']]
    joined.append({
        **p,
        'source': r.get('source', 'human'),
        'model': r.get('model', 'human'),
        'category': r.get('category', 'نامشخص'),
        'num_words': r.get('num_words', 0),
    })

# دقت کل
correct = sum(1 for p in joined if p['correct'])
overall_acc = correct / len(joined)
print(f'دقت کل: {overall_acc:.4f}')

In [ ]:
# تفکیک بر اساس منبع (انسان + هر سه مولد)
by_model = defaultdict(list)
for p in joined:
    key = 'human' if p['source'] == 'human' else p['model']
    by_model[key].append(p)

by_model_stats = {}
for m, items in by_model.items():
    accs = [x['correct'] for x in items]
    confs = [x.get('logit_margin', 0) for x in items]
    by_model_stats[m] = {
        'accuracy': float(np.mean(accs)),
        'count': len(items),
        'mean_confidence': float(np.mean(confs))
    }
    print(f'  {m}: دقت={by_model_stats[m]["accuracy"]:.4f}  (n={len(items)})')

In [ ]:
# تفکیک بر اساس حوزه موضوعی
by_cat = defaultdict(list)
for p in joined:
    by_cat[p['category']].append(p['correct'])

by_cat_stats = {c: {'accuracy': float(np.mean(v)), 'count': len(v)} for c, v in by_cat.items()}
print('\nبر اساس حوزه:')
for c, s in by_cat_stats.items():
    print(f'  {c}: {s["accuracy"]:.4f} (n={s["count"]})')

In [ ]:
# تفکیک بر اساس طول
buckets = [('۴۰–۵۰', 40, 50), ('۵۰–۸۰', 50, 80), ('۸۰–۱۲۰', 80, 120), ('۱۲۰+', 120, 10000)]
by_len_stats = {}
for label, lo, hi in buckets:
    items = [p['correct'] for p in joined if lo <= p['num_words'] < hi]
    if items:
        by_len_stats[label] = {'accuracy': float(np.mean(items)), 'count': len(items)}

print('\nبر اساس طول:')
for label, s in by_len_stats.items():
    print(f'  {label}: {s["accuracy"]:.4f} (n={s["count"]})')

In [ ]:
# ذخیره
from datetime import datetime
out = {
    'experiment': 'error_analysis',
    'timestamp': datetime.now().isoformat(),
    'overall_accuracy': overall_acc,
    'by_model': by_model_stats,
    'by_category': by_cat_stats,
    'by_length': by_len_stats
}
with open(RESULTS_DIR / 'error_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(out, f, ensure_ascii=False, indent=2)
print('✓ ذخیره شد در results/error_analysis.json')